# Advanced Analysis Notebook

This notebook performs advanced analysis using shared data from task_values.

In [ ]:
import pandas as pd
import numpy as np
import json

In [ ]:
# Get shared data from input parameters (passed from previous notebook's task_values)
mean_x = float(dbutils.widgets.get("raw_data_mean_x", "0"))
mean_y = float(dbutils.widgets.get("raw_data_mean_y", "0")) 
std_x = float(dbutils.widgets.get("raw_data_std_x", "1"))
std_y = float(dbutils.widgets.get("raw_data_std_y", "1"))
data_size = int(dbutils.widgets.get("data_size_actual", "0"))

# Handle category_distribution JSON parameter
category_dist_str = dbutils.widgets.get("category_distribution", "{}")
try:
    category_dist = json.loads(category_dist_str) if category_dist_str != "{}" else {}
except:
    category_dist = {}

print("📊 Using data passed from validation step:")
print(f"  Mean X: {mean_x:.3f}, Mean Y: {mean_y:.3f}")
print(f"  Std X: {std_x:.3f}, Std Y: {std_y:.3f}")
print(f"  Data size: {data_size}")
print(f"  Category distribution: {category_dist}")

In [ ]:
# Advanced statistical analysis
print("🔬 Performing ADVANCED analysis...")

# Calculate advanced metrics using shared data
coefficient_of_variation_x = std_x / abs(mean_x) if mean_x != 0 else 0
coefficient_of_variation_y = std_y / abs(mean_y) if mean_y != 0 else 0
signal_to_noise_x = abs(mean_x) / std_x if std_x != 0 else 0
signal_to_noise_y = abs(mean_y) / std_y if std_y != 0 else 0

# Advanced category analysis
category_entropy = 0
total_samples = sum(category_dist.values())
for count in category_dist.values():
    if count > 0:
        prob = count / total_samples
        category_entropy -= prob * np.log2(prob)

print(f"  Coefficient of Variation X: {coefficient_of_variation_x:.3f}")
print(f"  Coefficient of Variation Y: {coefficient_of_variation_y:.3f}")
print(f"  Signal-to-Noise Ratio X: {signal_to_noise_x:.3f}")
print(f"  Signal-to-Noise Ratio Y: {signal_to_noise_y:.3f}")
print(f"  Category Entropy: {category_entropy:.3f}")

In [ ]:
# Share advanced metrics via task_values for downstream notebooks
dbutils.jobs.taskValues.set("cv_x", coefficient_of_variation_x)
dbutils.jobs.taskValues.set("cv_y", coefficient_of_variation_y)
dbutils.jobs.taskValues.set("snr_x", signal_to_noise_x)
dbutils.jobs.taskValues.set("snr_y", signal_to_noise_y)
dbutils.jobs.taskValues.set("category_entropy", category_entropy)
dbutils.jobs.taskValues.set("analysis_type", "advanced")

print("✅ Shared advanced metrics via task_values")

In [ ]:
# Determine recommendation for next step
high_quality_data = (
    coefficient_of_variation_x < 2.0 and 
    coefficient_of_variation_y < 2.0 and
    category_entropy > 1.0  # Good category diversity
)

if high_quality_data:
    recommendation = "PROCEED_TO_MODELING"
    confidence = "HIGH" 
    print("🚀 HIGH quality data - recommend proceeding to modeling!")
else:
    recommendation = "ADDITIONAL_PREPROCESSING"
    confidence = "MEDIUM"
    print("⚠️  Data needs additional preprocessing before modeling")

# Return recommendation for workflow decision
advanced_results = {
    "analysis_type": "advanced",
    "recommendation": recommendation,
    "confidence": confidence,
    "data_quality_score": (signal_to_noise_x + signal_to_noise_y) / 2,
    "advanced_metrics": {
        "cv_x": coefficient_of_variation_x,
        "cv_y": coefficient_of_variation_y,
        "category_entropy": category_entropy
    }
}

print(f"🎯 Advanced Analysis Result: {recommendation} ({confidence} confidence)")

# This exit_value will be used by workflow for next step decision
dbutils.notebook.exit(advanced_results)